In [1]:
# Exercise 5 - Build RAG without langchain + Exercise 6 - send the prompt to LLM

document = """
RAG stands for Retrieval Augmented Generation.

RAG improves LLM responses by retrieving relevant information from external documents.

Embeddings convert text into vectors.

Vector databases store embeddings and allow similarity search.
"""
# chunking the document : for now doing it manually
chunks = [
    "RAG stands for Retrieval Augmented Generation.",
    "RAG improves LLM responses by retrieving relevant information from external documents.",
    "Embeddings convert text into vectors.",
    "Vector databases store embeddings and allow similarity search."
]


In [2]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer("all-MiniLM-L6-v2")
from sklearn.metrics.pairwise import cosine_similarity

chunk_embeddings = model.encode(chunks)

def retrieve(query, k=2):
    query_embedding = model.encode([query])

    similarities = cosine_similarity(
        query_embedding,
        chunk_embeddings
    )[0]

    top_indices = similarities.argsort()[-k:][::-1]
    return [chunks[i] for i in top_indices]


W0627 19:54:28.000000 27872 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.
C:\Users\uniqu\AppData\Local\Programs\Python\Python311\Lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [3]:
def build_prompt(query,retrieved_chunks):
    context = "\n".join(retrieved_chunks)
    prompt = f"""
context: 
{context}
    
Question:
{query}

Answer only using the context above
"""

    return prompt

In [4]:
query = "What do embeddings do ?"
retrieved_chunks = retrieve(query)
prompt = build_prompt(
    query,
    retrieved_chunks
)
print(prompt)


context: 
Embeddings convert text into vectors.
Vector databases store embeddings and allow similarity search.

Question:
What do embeddings do ?

Answer only using the context above



In [ ]:
from google import genai

client = genai.Client(api_key="GEMINI_API_KEY")

response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=prompt
    )

print(response.text)

Embeddings convert text into vectors.
